In [1]:
import joblib
from dask.distributed import Client as DaskClient
from sklearn.ensemble import RandomForestClassifier

from utils import BBOX, BBOX_GDF, DATETIME, classes, load_data, training_data

In [2]:
%reload_ext autoreload
%autoreload 2

## Find and load S2 data

Load data and set up your array to use for prediction

In [3]:
# Create local dask cluster to improve data load time. Only run this once.
dask_client = DaskClient(n_workers=1, threads_per_worker=16, memory_limit="16GB")

In [4]:
BBOX_GDF.explore()

In [5]:
merged = load_data(BBOX, DATETIME)

Found Sentinel-2 items: 4
Found DEM items: 1
Found Sentinel-1 items: 4


In [6]:
merged

<xarray.Dataset> Size: 2GB
Dimensions:        (y: 3370, x: 5567)
Coordinates:
  * y              (y) float64 27kB -1.98e+06 -1.98e+06 ... -2.014e+06
  * x              (x) float64 45kB 3.124e+06 3.124e+06 ... 3.179e+06 3.179e+06
    spatial_ref    int32 4B 3832
    time           datetime64[us] 8B 2022-01-01
Data variables: (12/29)
    blue           (y, x) uint16 38MB dask.array<chunksize=(2048, 2048), meta=np.ndarray>
    green          (y, x) uint16 38MB dask.array<chunksize=(2048, 2048), meta=np.ndarray>
    red            (y, x) uint16 38MB dask.array<chunksize=(2048, 2048), meta=np.ndarray>
    rededge1       (y, x) uint16 38MB dask.array<chunksize=(2048, 2048), meta=np.ndarray>
    rededge2       (y, x) uint16 38MB dask.array<chunksize=(2048, 2048), meta=np.ndarray>
    rededge3       (y, x) uint16 38MB dask.array<chunksize=(2048, 2048), meta=np.ndarray>
    ...             ...
    northness      (y, x) float32 75MB dask.array<chunksize=(2048, 2048), meta=np.ndarray>
    eastness       (y, x) float32 75MB dask.array<chunksize=(2048, 2048), meta=np.ndarray>
    elevation_std  (y, x) float32 75MB dask.array<chunksize=(2052, 2052), meta=np.ndarray>
    mean_vv        (y, x) float32 75MB dask.array<chunksize=(2048, 2048), meta=np.ndarray>
    mean_vh        (y, x) float32 75MB dask.array<chunksize=(2048, 2048), meta=np.ndarray>
    mean_vv_vh     (y, x) float32 75MB dask.array<chunksize=(2048, 2048), meta=np.ndarray>

## Train and predict

When you change your training data, you can re-train and predict here.

In [7]:
print(f"Number of training samples: {len(training_data)}")

# print(training_data.crs)
# print(merged.odc.crs)

# Reproject 4326 training data to 3832 like the main dataset
training_data = training_data.to_crs(merged.odc.crs)

print("Distribution of lulc_code in the training data:")
print(training_data["lulc_code"].value_counts().sort_index())

# Colour points by class, legend shows "code: label"
labels = {code: f"{code}: {name}" for code, name, _ in classes}
training_data.assign(lulc_class=training_data["lulc_code"].map(labels)).explore(
    column="lulc_class",
    categorical=True,
    categories=list(labels.values()),
    cmap=[color for *_, color in classes],
    legend=True,
)

Number of training samples: 254
Distribution of lulc_code in the training data:
lulc_code
1    83
2    75
3     9
4    15
5    18
7    32
8    22
Name: count, dtype: int64


In [8]:
# Sample merged at each training point (nearest pixel)
points = training_data.get_coordinates().to_xarray()
samples = merged.sel(
    x=points.x,
    y=points.y,
    method="nearest",
    # Finally load the merged data. Until now it was lazy
).compute()
training_array = training_data.join(samples.to_dataframe())

# Keep only the lulc code and model features
features = list(merged.data_vars)
training_array = training_array[["lulc_code", *features]]

# Drop points that fall on NaN (masked) pixels
n_points = len(training_array)
training_array = training_array.dropna()
print(
    f"Training array: {training_array.shape}, dropped {n_points - len(training_array)} NaN rows"
)

/Users/wj/Projects/dep/minerals-model/.venv/lib/python3.15/site-packages/rasterio/warp.py:390: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dest = _reproject(
2026-10-26 10:45:51,925 - distributed.worker.memory - WARNING - Unmanaged memory use is high. This may indicate a memory leak or the memory may not be released to the OS; see https://distributed.dask.org/en/latest/worker-memory.html#memory-not-released-back-to-the-os for more information. -- Unmanaged memory: 10.77 GiB -- Worker memory limit: 14.90 GiB


Training array: (253, 30), dropped 1 NaN rows


In [9]:
training_array.to_csv("training_array.csv", index=False)
training_array

,lulc_code,blue,green,red,rededge1,rededge2,rededge3,nir,swir16,swir22,...,ndwi,ndre,elevation,slope,northness,eastness,elevation_std,mean_vv,mean_vh,mean_vv_vh
0,1,811,934,936,1194,1518,1656,1236,1195,873,...,-0.139171,0.017284,5.000000,0.000000,0.000000,0.000000,4.394247,0.052183,0.017979,2.902464
1,2,573,848,501,1289,3659,4513,4642,2144,1050,...,-0.691075,0.565335,375.317383,31.472502,0.518942,0.854810,10.667843,0.201796,0.039763,5.074977
2,2,413,643,323,1019,3803,5025,5047,1926,839,...,-0.773989,0.664029,180.775040,39.766327,0.769710,-0.638394,13.965274,0.236095,0.069069,3.418253
3,8,647,923,664,1286,2815,3302,3492,1822,1058,...,-0.581880,0.461699,15.229146,6.725863,0.572591,0.819841,1.685645,0.229282,0.044159,5.192220
4,2,409,701,365,1158,3391,4133,4219,1936,882,...,-0.715041,0.569277,234.587357,8.812180,-0.962956,0.269660,6.413419,0.641541,0.125661,5.105337
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
249,8,728,782,777,926,1034,1117,1073,1106,835,...,-0.156873,0.073537,16.469902,0.000000,0.000000,0.000000,4.649040,0.154924,0.028212,5.491388
250,8,767,1082,1001,1545,3147,3695,3046,2174,1277,...,-0.475775,0.326944,30.299189,10.744373,0.996674,0.081493,3.930384,0.308307,0.074245,4.152589
251,4,686,1018,940,1435,3209,3768,3133,2165,1233,...,-0.509516,0.371716,200.845093,3.307093,0.211276,-0.977426,2.246981,0.143315,0.036147,3.964744
252,4,721,1025,747,1512,2965,3428,3467,1869,1006,...,-0.543633,0.392649,180.510117,2.984920,0.000000,-1.000000,2.799363,0.176239,0.066094,2.666483


In [13]:
classifier = RandomForestClassifier(
    n_estimators=500,
    max_depth=None,
    min_samples_leaf=1,
    class_weight="balanced",
    n_jobs=-1,
    random_state=42,
    oob_score=True,
)

# DataFrame keeps feature names on the model
model = classifier.fit(training_array[features], training_array["lulc_code"])

In [14]:
print(classifier.oob_score_)

0.7470355731225297


In [15]:
joblib.dump(model, "test_model.dump")

['test_model.dump']

In [16]:
# Print feature importances against column headings
fields_importances = sorted(
    zip(features, classifier.feature_importances_),
    key=lambda x: x[1],
    reverse=True,
)

for i in fields_importances:
    print(f"{i[0]:<11}| {i[1]:.2f}")

blue       | 0.07
swir22     | 0.07
green      | 0.06
ndwi       | 0.06
smad       | 0.05
elevation  | 0.05
elevation_std| 0.05
swir16     | 0.04
emad       | 0.04
red        | 0.04
nir        | 0.04
ndre       | 0.03
ndvi       | 0.03
rededge1   | 0.03
bsi        | 0.03
ndbi       | 0.03
bcmad      | 0.03
rededge2   | 0.03
savi       | 0.03
rededge3   | 0.02
mean_vv_vh | 0.02
slope      | 0.02
evi        | 0.02
ndmi       | 0.02
mndwi      | 0.02
mean_vh    | 0.01
mean_vv    | 0.01
northness  | 0.01
eastness   | 0.01
